# Dữ liệu và kiểm tra điều kiện đề bài

Chương này tải hai tập dữ liệu dùng trong đồ án, mô tả cấu trúc và kiểm tra từng điều kiện mà đề bài đặt ra cho mỗi loại dữ liệu.

| Phần | Tập dữ liệu | Nguồn |
|---|---|---|
| (a) Bảng | Adult Census Income | [scikit-learn/adult-census-income](https://huggingface.co/datasets/scikit-learn/adult-census-income) |
| (b) Văn bản | IMDB Movie Reviews | [stanfordnlp/imdb](https://huggingface.co/datasets/stanfordnlp/imdb) |

Code tải dữ liệu nằm trong `book/utils/data_loader.py`. Lần chạy đầu tiên tải từ Hugging Face và lưu file parquet vào `data/raw/`; các lần sau đọc lại từ file cục bộ.

In [1]:
import pandas as pd

from utils.config import RANDOM_STATE, RAW_DIR, set_seed
from utils.data_loader import load_adult, load_imdb

set_seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
print("Thư mục dữ liệu thô:", RAW_DIR)

Thư mục dữ liệu thô: Z:\ML\dm-preprocessing\data\raw


## Phần (a): Adult Census Income

Dữ liệu trích từ cuộc điều tra dân số Mỹ năm 1994. Bài toán gốc là dự đoán thu nhập năm của một người có vượt 50.000 USD hay không. Trong bản gốc, giá trị thiếu được ghi bằng ký tự `?`, vì vậy cần đổi sang `NaN` trước khi đếm.

In [2]:
adult_raw = load_adult()
adult = adult_raw.replace("?", pd.NA)
print("Kích thước:", adult.shape)
adult.head()

Kích thước: (32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,NaN,77053,HS-grad,9,Widowed,NaN,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,NaN,186061,Some-college,10,Widowed,NaN,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [3]:
TARGET = "income"
num_cols = adult.drop(columns=TARGET).select_dtypes("number").columns.tolist()
cat_cols = [c for c in adult.columns if c not in num_cols + [TARGET]]

adult_schema = pd.DataFrame({
    "kiểu": ["số" if c in num_cols else ("nhãn" if c == TARGET else "phân loại") for c in adult.columns],
    "số giá trị khác nhau": adult.nunique(),
    "số ô thiếu": adult.isna().sum(),
    "% thiếu": (adult.isna().mean() * 100).round(2),
}, index=adult.columns)
adult_schema

,kiểu,số giá trị khác nhau,số ô thiếu,% thiếu
age,số,73,0,0.00
workclass,phân loại,8,1836,5.64
fnlwgt,số,21648,0,0.00
education,phân loại,16,0,0.00
education.num,số,16,0,0.00
marital.status,phân loại,7,0,0.00
occupation,phân loại,14,1843,5.66
relationship,phân loại,6,0,0.00
race,phân loại,5,0,0.00
sex,phân loại,2,0,0.00


In [4]:
adult_target_dist = adult[TARGET].value_counts(normalize=True).round(4)
print("Phân phối nhãn:")
print(adult_target_dist)

Phân phối nhãn:
income
<=50K    0.7592
>50K     0.2408
Name: proportion, dtype: float64


### Kiểm tra điều kiện đề bài cho dữ liệu bảng

Đề yêu cầu: ít nhất 10 thuộc tính, ít nhất 10.000 bản ghi, có cả thuộc tính số và phân loại, và có ít nhất một thuộc tính thiếu từ 5% trở lên.

In [5]:
max_missing_col = adult_schema["% thiếu"].idxmax()
adult_checks = pd.DataFrame({
    "điều kiện": ["≥ 10 thuộc tính", "≥ 10.000 bản ghi", "có thuộc tính số", "có thuộc tính phân loại",
                  "≥ 1 thuộc tính thiếu ≥ 5%"],
    "giá trị thực tế": [len(num_cols) + len(cat_cols), len(adult), len(num_cols), len(cat_cols),
                        f"{max_missing_col}: {adult_schema.loc[max_missing_col, '% thiếu']}%"],
    "đạt": [len(num_cols) + len(cat_cols) >= 10, len(adult) >= 10_000, len(num_cols) > 0, len(cat_cols) > 0,
            adult_schema["% thiếu"].max() >= 5],
})
adult_checks

,điều kiện,giá trị thực tế,đạt
0,≥ 10 thuộc tính,14,True
1,≥ 10.000 bản ghi,32561,True
2,có thuộc tính số,6,True
3,có thuộc tính phân loại,8,True
4,≥ 1 thuộc tính thiếu ≥ 5%,occupation: 5.66%,True


## Phần (b): IMDB Movie Reviews

Tập dữ liệu của Maas et al. (2011) gồm 50.000 review phim tiếng Anh có nhãn cảm xúc nhị phân, chia sẵn 25.000 cho train và 25.000 cho test. Đồ án giữ nguyên cách chia này để tính tỷ lệ OOV trên tập test ở chương tokenization.

In [6]:
imdb = load_imdb()
print("Kích thước:", imdb.shape)
imdb.head()

Kích thước: (50000, 3)


,text,label,split
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,train
1,"""I Am Curious: Yellow"" is a risible and preten...",0,train
2,If only to avoid making this type of film in t...,0,train
3,This film was probably inspired by Godard's Ma...,0,train
4,"Oh, brother...after hearing about this ridicul...",0,train


In [7]:
imdb_label_split = pd.crosstab(imdb["split"], imdb["label"], margins=True)
imdb_label_split.columns = ["negative (0)", "positive (1)", "tổng"]
imdb_label_split

,negative (0),positive (1),tổng
split,,,
test,12500,12500,25000
train,12500,12500,25000
All,25000,25000,50000


### Kiểm tra điều kiện đề bài cho dữ liệu văn bản

Đề yêu cầu: ít nhất 10.000 mẫu và ít nhất 2 nhãn phân loại.

In [8]:
imdb_checks = pd.DataFrame({
    "điều kiện": ["≥ 10.000 mẫu", "≥ 2 nhãn"],
    "giá trị thực tế": [len(imdb), imdb["label"].nunique()],
    "đạt": [len(imdb) >= 10_000, imdb["label"].nunique() >= 2],
})
imdb_checks

,điều kiện,giá trị thực tế,đạt
0,≥ 10.000 mẫu,50000,True
1,≥ 2 nhãn,2,True


## Nhận xét

- Adult Census Income thỏa mọi điều kiện của phần (a). Ba thuộc tính phân loại có giá trị thiếu, trong đó tỷ lệ thiếu cao nhất vượt ngưỡng 5%. Nhãn mất cân bằng khoảng 76/24, đủ để thực hiện phần nâng cao về xử lý mất cân bằng lớp. Thuộc tính `native.country` có hơn 20 giá trị khác nhau nên phù hợp để thử Binary Encoding.
- IMDB thỏa điều kiện phần (b) với 50.000 mẫu và hai nhãn cân bằng tuyệt đối.